In [3]:
%pip install pennylane pennylane-lightning qiskit qiskit-aer matplotlib scipy pyqsp --break-system-packages

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import pennylane as qml
import numpy as np


def block_encoding_test_1_ancilla():
    # 1. Define the target non-unitary matrix A
    # Ensure spectral norm <= 1 (which it is for 0.5 and 0.3)
    A = np.array([[0.5, 0.0], [0.0, 0.3]])

    print("1. Original Target Matrix A:")
    print(A)
    print("-" * 40)

    # 2. Create the Block Encoding Operator
    # A is a 2x2 matrix (requires 1 qubit).
    # To make it unitary, we need at least 1 extra "ancilla" qubit.
    # Total: 2 qubits (wires 0 and 1). We'll treat wire 0 as the ancilla.
    op = qml.BlockEncode(A, wires=[0, 1])

    # 3. Get the full mathematical representation of the resulting Unitary
    U = qml.matrix(op)

    print("2. Full 4x4 Unitary Matrix U from PennyLane:")
    print(np.round(U, 3))  # Rounding for cleaner console output
    print("-" * 40)

    # 4. Extract the top-left 2x2 block
    # In a 4x4 matrix representing wires [0, 1], the top-left 2x2 block
    # represents the subspace where wire 0 (the ancilla) is in the |0> state.
    top_left_block = U[0:2, 0:2]

    print("3. Extracted Top-Left Block (Ancilla = |0>):")
    print(np.round(top_left_block, 3))
    print("-" * 40)

    # 5. Verify the match
    # np.allclose checks if all elements are equal within a small tolerance
    is_match = np.allclose(A, top_left_block)
    print(f"SUCCESS: Does the extracted block match A? --> {is_match}")


block_encoding_test_1_ancilla()

1. Original Target Matrix A:
[[0.5 0. ]
 [0.  0.3]]
----------------------------------------
2. Full 4x4 Unitary Matrix U from PennyLane:
[[ 0.5    0.     0.866  0.   ]
 [ 0.     0.3    0.     0.954]
 [ 0.866  0.    -0.5   -0.   ]
 [ 0.     0.954 -0.    -0.3  ]]
----------------------------------------
3. Extracted Top-Left Block (Ancilla = |0>):
[[0.5 0. ]
 [0.  0.3]]
----------------------------------------
SUCCESS: Does the extracted block match A? --> True


In [ ]:
import pyqsp
from pyqsp.poly import PolySign
from pyqsp.angle_sequence import QuantumSignalProcessingPhases
import numpy as np


def generate_oaa_phases(degree=15, kappa=10):
    """
    Generates QSP phase angles for Oblivious Amplitude Amplification.
    """
    print(f"--- Generating QSP Angles for OAA (Degree={degree}) ---")

    # 1. Generate the polynomial coefficients
    # The API now uses `delta` instead of `kappa`, and `return_scale` instead of `return_coef`.
    pcoefs, scale = PolySign().generate(degree=degree, delta=kappa, return_scale=True)

    print(f"1. Generated Polynomial Coefficients (Scale: {scale}):")
    print(np.round(pcoefs, 4))
    print("-" * 50)

    # 2. Compute the QSP Phase Angles
    # We pass the coefficients to the phase angle generator.
    try:
        # We specify the Wx signal operator to match PennyLane's QSVT convention
        phases = QuantumSignalProcessingPhases(pcoefs, signal_operator="Wx")
    except Exception as e:
        print(f"Error generating phases: {e}")
        return None

    print(f"2. Successfully computed {len(phases)} QSP phase angles (in radians):")
    formatted_phases = [round(float(p), 4) for p in phases]
    print(formatted_phases)
    print("-" * 50)

    return phases


if __name__ == "__main__":
    angles = generate_oaa_phases(degree=15, kappa=10)

    if angles is not None:
        print("\nNext step: Feed this array of angles into PennyLane's qml.qsvt()!")

--- Generating QSP Angles for OAA (Degree=15) ---
[pyqsp.poly.PolySign] degree=15, delta=10
[PolyTaylorSeries] (Cheb) max 0.8684663045726337 is at -0.5617068413013444: normalizing
[PolyTaylorSeries] (Cheb) average error = 0.030242022153254906 in the domain [-1, 1] using degree 15
1. Generated Polynomial Coefficients (Scale: 0.8684663045726337):
[ 0.      1.103   0.     -0.3604  0.      0.2077  0.     -0.1397  0.
  0.1003  0.     -0.0743  0.      0.0558  0.     -0.0421]
--------------------------------------------------
2. Successfully computed 16 QSP phase angles (in radians):
[-1.6119, 0.0516, -0.0655, 0.0849, -0.114, 0.164, -0.2736, 0.702, 0.702, -0.2736, 0.164, -0.114, 0.0849, -0.0655, 0.0516, -0.0411]
--------------------------------------------------

Next step: Feed this array of angles into PennyLane's qml.qsvt()!


In [87]:
import pennylane as qml
import numpy as np
from pyqsp.poly import PolySign
from pyqsp.angle_sequence import QuantumSignalProcessingPhases


def run_algorithm1_integration_test(degree=15, delta=10):
    """
    Executes a full end-to-end test of Algorithm 1's space-time tradeoff,
    comparing a naive block encoding against a QSVT-amplified block encoding.
    """
    print(
        f"--- Starting Algorithm 1 Integration Test (Degree={degree}, Delta={delta}) ---\n"
    )

    # 1. Define the Target Matrix
    A = np.array([[0.5, 0.0], [0.0, 0.3]])

    # 2. Generate the QSP Angles
    try:
        pcoefs, scale = PolySign().generate(
            degree=degree, delta=delta, return_scale=True
        )
        # We specify "Wx" so the phases match standard rotation conventions
        phases = QuantumSignalProcessingPhases(pcoefs, signal_operator="Wx")
    except Exception as e:
        print(f"Error generating phases: {e}")
        return

    # 3. Set up the PennyLane Quantum Device
    dev = qml.device("default.qubit", wires=2)

    # 4. Define the Sub-Circuits locally
    @qml.qnode(dev)
    def vanilla_circuit():
        qml.BlockEncode(A, wires=[0, 1])
        return qml.probs(wires=0)

    @qml.qnode(dev)
    def qsvt_circuit():
        # A. Explicitly define the block encoding operator
        U = qml.BlockEncode(A, wires=[0, 1])

        # B. Explicitly define the projector-controlled phase shifts
        # dim=2 because our target matrix A is 2x2.
        projectors = [qml.PCPhase(phi, dim=2, wires=[0, 1]) for phi in phases]

        # C. Apply the uppercase QSVT template
        qml.QSVT(U, projectors)

        return qml.probs(wires=0)

    # 5. Execute and Output Results
    print("--- 1. Vanilla Block Encoding ---")
    vanilla_probs = vanilla_circuit()
    print(
        f"Prob(Ancilla = |0>): {vanilla_probs[0]:.4f}"
    )
    print(f"Prob(Ancilla = |1>): {vanilla_probs[1]:.4f}\n")

    print("--- 2. Alg 1: QSVT Amplitude Amplification ---")
    qsvt_probs = qsvt_circuit()
    print(f"Prob(Ancilla = |0>): {qsvt_probs[0]:.4f}")
    print(f"Prob(Ancilla = |1>): {qsvt_probs[1]:.4f}\n")



# Run the function
if __name__ == "__main__":
    run_algorithm1_integration_test(degree=47, delta=2)

--- Starting Algorithm 1 Integration Test (Degree=47, Delta=2) ---

[pyqsp.poly.PolySign] degree=47, delta=2
[PolyTaylorSeries] (Cheb) max 0.9042297471189998 is at -1.0: normalizing
[PolyTaylorSeries] (Cheb) average error = 1.2616643840779317e-15 in the domain [-1, 1] using degree 47
--- 1. Vanilla Block Encoding ---
Prob(Ancilla = |0>): 0.2500
Prob(Ancilla = |1>): 0.7500

--- 2. Alg 1: QSVT Amplitude Amplification ---
Prob(Ancilla = |0>): 0.7516
Prob(Ancilla = |1>): 0.2484

